In [1]:
#------------------------------------------------------------------------------------------------------#
#
# Code:      CC01_CCAR_C03_AgenticAI_model_scoring_01.ipynb
#
# Objective: Step C03: Use Agentic AI to score vintages of 201605, 201606, 201607, ..., 201801.
#                      It is the medium level of Agentic AI.
#
#            Jingru Chen
#            2026-03-22
#
#----------------------------------------------------------------------------------------------------#

In [2]:
# In this version, added a Performance Memory buffer and a Decision Layer (_check_model_drift) that acts as the Agent's "conscience."

# Step 0: Upload libraries

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from pydantic import BaseModel, Field

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [4]:
from datetime import datetime
from zoneinfo import ZoneInfo

run_date="2026-03-22"

start = datetime.now( ZoneInfo("America/New_York"))

print( start.strftime("%Y-%m-%d %H:%M:%S %Z"))     # 2026-03-17 17:34:58 EDT
print( start.strftime("%Y-%m-%d %I:%M:%S %p %Z"))  # 2026-03-17 05:34:58 PM EDT

2026-03-22 22:54:01 EDT
2026-03-22 10:54:01 PM EDT


In [5]:
myout= "/content/sample_data"

In [6]:
pwd

'/content'

In [7]:
cd /content/sample_data/

/content/sample_data


In [8]:
ls -ltr

total 55504
-rwxr-xr-x 1 root root      962 Jan  1  2000 README.md*
-rwxr-xr-x 1 root root     1697 Jan  1  2000 anscombe.json*
-rw-r--r-- 1 root root  1706430 Mar 17 17:58 california_housing_train.csv
-rw-r--r-- 1 root root   301141 Mar 17 17:58 california_housing_test.csv
-rw-r--r-- 1 root root 36523880 Mar 17 17:58 mnist_train_small.csv
-rw-r--r-- 1 root root 18289443 Mar 17 17:58 mnist_test.csv


In [13]:
x_list=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate', 'loan_term_months',
        'delta_Unemployment1',
       'delta_Mortgage_rate1', 'delta_House_Price_Index__Level1',
       'delta_Unemployment3', 'delta_Mortgage_rate3',
       'delta_House_Price_Index__Level3', 'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12', 'delta_Mortgage_rate12',
       'delta_House_Price_Index__Level12', 'delta_Unemployment24',
       'delta_Mortgage_rate24', 'delta_House_Price_Index__Level24']

x_list_v2=['original_balance', 'credit_score_orig', 'loan_to_value_orig', 'interest_rate',
        'delta_Unemployment1',
       'delta_Unemployment3',
       'delta_Unemployment6',
       'delta_Mortgage_rate6', 'delta_House_Price_Index__Level6',
       'delta_Unemployment12',
       'delta_Unemployment24' ]

# y_list= ['flag_default']

pd_model = 'ccar_pd_model_2026-03-22.pkl'
pd_input_file= "/CCAR_Mortgage_data_for_model_DEV_20260319_01.csv"
# pd_vintage = 201604
pd_x_list = x_list_v2
pd_y = 'flag_default'
pd_threshold = 0.5


# Step 1: The "Decision Layer" Agentic Logic

In [23]:
class CCARProductionAgent:
    """
    An Autonomous Agentic AI for CCAR Production.
    Features: Persistent State, Decision Layer, and Drift Protection.
    """
    def __init__(self, agent_name, model_path, input_file, x_list, y_target, threshold, output_path="./"):
        self.agent_name = agent_name
        self.model = joblib.load(model_path)
        self.input_file = input_file
        self.x_list = x_list
        self.y_target = y_target
        self.threshold = threshold
        self.output_path = output_path

        # --- AGENT MEMORY ---
        self.recall_history = []
        self.master_report = None
        self.is_aborted = False

    def _check_model_drift(self, current_recall, vintage):
        """
        DECISION LAYER: Evaluates if the Agent should continue or stop.
        Logic: If Recall < 0.60 for 3 consecutive runs, trigger a Kill Switch.
        """
        self.recall_history.append(current_recall)

        # Look at the last 3 performance observations
        if len(self.recall_history) >= 3:
            recent_performance = self.recall_history[-3:]
            if all(r < 0.60 for r in recent_performance):
                self.is_aborted = True
                print(f"\n [CRITICAL ALERT - {self.agent_name}]")
                print(f"Model Drift Detected at vintage {vintage}!")
                print(f"Recall has dropped below 60% for 3 consecutive months: {recent_performance}")
                print("ABORTING PRODUCTION RUN TO PREVENT REGULATORY MISREPORTING.")

    def _scoring_action(self, vintage: str):
        if self.is_aborted:
            return {"vintage": vintage, "status": "ABORTED (Drift Detected)"}

        try:
            df = pd.read_csv(self.output_path + self.input_file)
            df_v = df.loc[df['report_yrmo'].astype(str) == str(vintage)].reset_index(drop=True)

            if df_v.empty:
                return {"vintage": vintage, "status": "Skipped (No Data)"}

            # Inference
            X, y = df_v[self.x_list], df_v[self.y_target]
            y_proba = self.model.predict_proba(X)[:, 1]
            y_pred = (y_proba >= self.threshold).astype(int)

            # Metrics
            tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
            rec = recall_score( y, y_pred, zero_division=0)
            precision= precision_score( y, y_pred, zero_division=0)

            # --- AGENT DECISION STEP ---
            self._check_model_drift(rec, vintage)

            return {
                "vintage": vintage,
                "status": "Success" if not self.is_aborted else "ABORTED",
                "count": len(df_v),
                "avg_pd": y_proba.mean(),
                "precision": precision,
                "recall": rec,
                "f1": f1_score(y, y_pred, zero_division=0),
                "TP": tp, "FN": fn, "TN": tn, "FP": fp
            }
        except Exception as e:
            return {"vintage": vintage, "status": f"Error: {str(e)}"}

    def run_production(self, start_date, end_date):
        vintages = pd.date_range(start=pd.to_datetime(start_date, format='%Y%m'),
                                 end=pd.to_datetime(end_date, format='%Y%m'),
                                 freq='MS').strftime('%Y%m').tolist()

        print(f" [{self.agent_name}] Starting production for {len(vintages)} vintages...")

        results = []
        for v in vintages:
            if self.is_aborted: break
            results.append(self._scoring_action(v))

        self.master_report = pd.DataFrame(results)
        return self.master_report

# --- MULTI-AGENT EXECUTION ---

# 1. Conservative Agent (Standard Operations)
agent_a = CCARProductionAgent("Agent_Conservative", pd_model, pd_input_file, pd_x_list, pd_y, 0.05)

# 2. Stress Test Agent (Adverse Scenarios)
agent_b = CCARProductionAgent("Agent_StressTest", pd_model, pd_input_file, pd_x_list, pd_y, 0.50)

print("====== 1.A Launching Parallel Agent Productions ======")
report_a = agent_a.run_production("201605", "201612")
report_b = agent_b.run_production("201605", "201612")

print("\n----------- 1.B Printout of report_a ----------\n", report_a )
print("\n----------- 1.C Printout of report_b ----------\n", report_b )

====== 1.A Launching Parallel Agent Productions ======
 [Agent_Conservative] Starting production for 8 vintages...
 [Agent_StressTest] Starting production for 8 vintages...

 [CRITICAL ALERT - Agent_StressTest]
Model Drift Detected at vintage 201607!
Recall has dropped below 60% for 3 consecutive months: [0.19230769230769232, 0.23076923076923078, 0.13636363636363635]
ABORTING PRODUCTION RUN TO PREVENT REGULATORY MISREPORTING.

----------- 1.B Printout of report_a ----------
   vintage   status  count    avg_pd  precision  recall        f1  TP  FN  TN  \
0  201605  Success   1157  0.307792   0.022472     1.0  0.043956  26   0   0   
1  201606  Success   1131  0.308470   0.022989     1.0  0.044944  26   0   0   
2  201607  Success   1105  0.324724   0.019910     1.0  0.039042  22   0   0   
3  201608  Success   1083  0.336038   0.024007     1.0  0.046889  26   0   0   
4  201609  Success   1057  0.324864   0.018921     1.0  0.037140  20   0   0   
5  201610  Success   1037  0.299116   0.

In [ ]:
from datetime import datetime
end = datetime.now(ZoneInfo("America/New_York"))
duration = end - start

print(f"Started:  {start}")
print(f"Finished: {end}")
print(f"\nDuration: {duration}")                    # 0:00:02.351234
print(f"Duration: {duration.total_seconds():.3f} seconds")